# Validation Set Extraction


## Imports


In [1]:
# Cell 1 — Setup
from pathlib import Path
import time

import numpy as np
from ase.io import write
from ase.io.trajectory import Trajectory



## Sampling method label


In [2]:
METHOD  = "FPS"               # "FPS", "Random", "Birch" ...
#METHOD  = "QBC"               # "FPS", "Random", "Birch" ...

## Build validation set


In [3]:
# sampled training set

# --- User inputs ---
selected_dir = Path("selected")
SRC_TRAJ = selected_dir / f"{METHOD}_selected.traj"
PERCENT = 20                   # e.g. 10 for 10%
SEED = 42                      # set None for non-deterministic

# Output directory
OUT_DIR = selected_dir / "valset"
OUT_DIR.mkdir(parents=True, exist_ok=True)
OUT_PREFIX = OUT_DIR / f"{METHOD}_selected_valset_{SEED}_{PERCENT}"

# Outputs
OUT_TRAJ = OUT_PREFIX.with_suffix(".traj")
OUT_XYZ = OUT_PREFIX.with_suffix(".xyz")
OUT_TXT = OUT_PREFIX.with_suffix(".txt")

# Sanity
assert 0 < PERCENT <= 100, "PERCENT must be in (0, 100]."
if not SRC_TRAJ.exists():
    candidates = sorted(selected_dir.glob("*_selected.traj"))
    if len(candidates) == 1:
        SRC_TRAJ = candidates[0]
    else:
        raise FileNotFoundError(f"Missing source trajectory: {SRC_TRAJ}")

# Inspect source trajectory
traj = Trajectory(str(SRC_TRAJ), mode="r")
n_frames = len(traj)
print(f"Source trajectory: {SRC_TRAJ}")
print(f"Number of frames in source: {n_frames}")

if n_frames == 0:
    raise ValueError(f"Source trajectory is empty: {SRC_TRAJ}")

k = max(1, int(round(PERCENT * n_frames / 100.0)))
if k > n_frames:
    raise ValueError(f"Requested {k} frames from a source with only {n_frames} frames.")

rng = np.random.default_rng(SEED)
indices = np.sort(rng.choice(n_frames, size=k, replace=False))

print(f"Frames that will be used: {n_frames}")
print(f"Sampling: {k} frames ({PERCENT}%)")
print("First 10 sampled indices:", indices[:10])

# Cell 3 — Write selected frames to .traj and .xyz
if OUT_TRAJ.exists():
    OUT_TRAJ.unlink()

with Trajectory(str(OUT_TRAJ), mode="w") as T:
    for i in indices:
        T.write(traj[i])

if OUT_XYZ.exists():
    OUT_XYZ.unlink()

first = True
for i in indices:
    write(str(OUT_XYZ), traj[i], append=not first, format="extxyz")
    first = False

print(f"Wrote: {OUT_TRAJ}")
print(f"Wrote: {OUT_XYZ}")

# Cell 4 — Manifest (.txt) with details for each selected frame
# Captures index, natoms, cell (a,b,c), pbc, energy

ts = time.strftime("%Y-%m-%d %H:%M:%S")

with OUT_TXT.open("w", encoding="utf-8") as f:
    f.write("# Validation set manifest\n")
    f.write(f"# Source: {SRC_TRAJ.resolve()}\n")
    f.write(f"# Created: {ts}\n")
    f.write(f"# Total frames: {n_frames}\n")
    f.write(f"# Percent: {PERCENT}\n")
    f.write(f"# Seed: {SEED}\n")
    f.write(f"# Selected frames: {k}\n\n")
    f.write("index\tnatoms\tcell_a\tcell_b\tcell_c\tpbc\tenergy\n")

    for i in indices:
        atoms = traj[i]
        cell = atoms.cell.lengths()
        pbc = tuple(bool(x) for x in atoms.pbc)
        energy = atoms.get_potential_energy()
        line = (
            f"{i}\t{len(atoms)}\t"
            f"{cell[0]:.6f}\t{cell[1]:.6f}\t{cell[2]:.6f}\t"
            f"{pbc}\t{energy}\n"
        )
        f.write(line)

print(f"Wrote: {OUT_TXT}")


Source trajectory: selected/FPS_selected.traj
Number of frames in source: 1006
Frames that will be used: 1006
Sampling: 201 frames (20%)
First 10 sampled indices: [ 6 22 29 33 34 37 53 55 57 66]
Wrote: selected/valset/FPS_selected_valset_42_20.traj
Wrote: selected/valset/FPS_selected_valset_42_20.xyz
Wrote: selected/valset/FPS_selected_valset_42_20.txt


## Build training set (complement)


In [4]:
# %%
# Cell 5 — Write training set (complement of validation set)
TRAIN_DIR = selected_dir / "trainset"
TRAIN_DIR.mkdir(parents=True, exist_ok=True)
TRAIN_PREFIX = TRAIN_DIR / f"{METHOD}_selected_trainset_{SEED}_{100-PERCENT}"

TRAIN_TRAJ = TRAIN_PREFIX.with_suffix(".traj")
TRAIN_XYZ = TRAIN_PREFIX.with_suffix(".xyz")
TRAIN_TXT = TRAIN_PREFIX.with_suffix(".txt")

# Build complement indices
all_indices = np.arange(n_frames)
train_indices = np.setdiff1d(all_indices, indices)

print(f"Training frames: {len(train_indices)} ({100-PERCENT}%)")

if TRAIN_TRAJ.exists():
    TRAIN_TRAJ.unlink()

with Trajectory(str(TRAIN_TRAJ), mode="w") as T:
    for i in train_indices:
        T.write(traj[i])

if TRAIN_XYZ.exists():
    TRAIN_XYZ.unlink()

first = True
for i in train_indices:
    write(str(TRAIN_XYZ), traj[i], append=not first, format="extxyz")
    first = False

# .txt manifest
ts = time.strftime("%Y-%m-%d %H:%M:%S")
with TRAIN_TXT.open("w", encoding="utf-8") as f:
    f.write("# Training set manifest\n")
    f.write(f"# Source: {SRC_TRAJ.resolve()}\n")
    f.write(f"# Created: {ts}\n")
    f.write(f"# Total frames: {n_frames}\n")
    f.write(f"# Percent kept: {100-PERCENT}\n")
    f.write(f"# Seed: {SEED}\n")
    f.write(f"# Training frames: {len(train_indices)}\n\n")
    f.write("index\tnatoms\tcell_a\tcell_b\tcell_c\tpbc\tenergy\n")

    for i in train_indices:
        atoms = traj[i]
        cell = atoms.cell.lengths()
        pbc = tuple(bool(x) for x in atoms.pbc)
        energy = atoms.get_potential_energy()
        line = (
            f"{i}\t{len(atoms)}\t"
            f"{cell[0]:.6f}\t{cell[1]:.6f}\t{cell[2]:.6f}\t"
            f"{pbc}\t{energy}\n"
        )
        f.write(line)

print(f"Wrote: {TRAIN_TRAJ}")
print(f"Wrote: {TRAIN_XYZ}")
print(f"Wrote: {TRAIN_TXT}")


Training frames: 805 (80%)
Wrote: selected/trainset/FPS_selected_trainset_42_80.traj
Wrote: selected/trainset/FPS_selected_trainset_42_80.xyz
Wrote: selected/trainset/FPS_selected_trainset_42_80.txt
